First, I'll profile the efficiency of matrix multiplication. Then I'll compare this to the efficiency using TF32 and using automatic mixed precision (AMP).

In [1]:
from time import time
import torch

In [2]:
warmup = 5
n = 10
N = 256
D = 1024

In [3]:
assert torch.cuda.is_available()

In [4]:
device = torch.device("cuda")
device_index = torch.cuda.current_device()
print(f"Device: {device.type}")
print(f"Currently selected GPU: {device_index}")
print(f"Name of current GPU: {torch.cuda.get_device_name(device_index)}")

Device: cuda
Currently selected GPU: 0
Name of current GPU: NVIDIA A40


# Matrix Multiplication Runtime

### FP32 Precision

In [5]:
X = torch.randn(N, D, D, device=device)

print(f"Data type: {X.dtype}")
print("----------------------")

print("Starting warmup")
for _ in range(warmup):
    C = torch.bmm(X, X)
torch.cuda.synchronize()
print("Warmup complete")

start_time = time()
for _ in range(n):
    C = torch.bmm(X, X)
torch.cuda.synchronize()
end_time = time()
total_time = end_time-start_time

print("FP32 Multiplication")
print("----------------------")
print(f"Time per matmul: {(total_time)/n:.3f} seconds")
print(f"Flops: {n*N*2*(D**3)/total_time/1e12:.2f} TFLOPs\n")

Data type: torch.float32
----------------------
Starting warmup
Warmup complete
FP32 Multiplication
----------------------
Time per matmul: 0.027 seconds
Flops: 20.02 TFLOPs



### TF32 Precision (19 bits)

In [6]:
torch.backends.cuda.matmul.allow_tf32 = True 
torch.backends.cudnn.allow_tf32 = True

In [7]:
X = torch.randn(N, D, D, device=device)

print(f"Data type: {X.dtype}")
print("----------------------")

print("Starting warmup")
for _ in range(warmup):
    C = torch.bmm(X, X)
torch.cuda.synchronize()
print("Warmup complete")

start_time = time()
for _ in range(n):
    C = torch.bmm(X, X)
torch.cuda.synchronize()
end_time = time()
total_time = end_time-start_time

print("TF32 Multiplication")
print("----------------------")
print(f"Time per matmul: {(total_time)/n:.3f} seconds")
print(f"Flops: {n*N*2*(D**3)/total_time/1e12:.2f} TFLOPs\n")

Data type: torch.float32
----------------------
Starting warmup
Warmup complete
TF32 Multiplication
----------------------
Time per matmul: 0.011 seconds
Flops: 49.96 TFLOPs



In [8]:
torch.backends.cuda.matmul.allow_tf32 = False
torch.backends.cudnn.allow_tf32 = False

### FP16 Precision

In [9]:
torch.set_default_dtype(torch.float16)

In [10]:
X = torch.randn(N, D, D, device=device)

print(f"Data type: {X.dtype}")
print("----------------------")

print("Starting warmup")
for _ in range(warmup):
    C = torch.bmm(X, X)
torch.cuda.synchronize()
print("Warmup complete")

start_time = time()
for _ in range(n):
    C = torch.bmm(X, X)
torch.cuda.synchronize()
end_time = time()
total_time = end_time-start_time

print("FP16 Multiplication")
print("----------------------")
print(f"Time per matmul: {(total_time)/n:.3f} seconds")
print(f"Flops: {n*N*2*(D**3)/total_time/1e12:.2f} TFLOPs\n")

Data type: torch.float16
----------------------
Starting warmup
Warmup complete
FP16 Multiplication
----------------------
Time per matmul: 0.007 seconds
Flops: 84.34 TFLOPs



In [11]:
torch.set_default_dtype(torch.float32)

### BF16 Precision

In [12]:
torch.set_default_dtype(torch.bfloat16)

In [13]:
X = torch.randn(N, D, D, device=device)

print(f"Data type: {X.dtype}")
print("----------------------")

print("Starting warmup")
for _ in range(warmup):
    C = torch.bmm(X, X)
torch.cuda.synchronize()
print("Warmup complete")

start_time = time()
for _ in range(n):
    C = torch.bmm(X, X)
torch.cuda.synchronize()
end_time = time()
total_time = end_time-start_time

print("BF16 Multiplication")
print("----------------------")
print(f"Time per matmul: {(total_time)/n:.3f} seconds")
print(f"Flops: {n*N*2*(D**3)/total_time/1e12:.2f} TFLOPs\n")

Data type: torch.bfloat16
----------------------
Starting warmup
Warmup complete
BF16 Multiplication
----------------------
Time per matmul: 0.006 seconds
Flops: 86.23 TFLOPs



In [14]:
torch.set_default_dtype(torch.float32)

### Automatic Mixed Precision (AMP)

In [15]:
X = torch.randn(N, D, D, device=device)

print(f"Data type: {X.dtype}")
print("----------------------")

with torch.autocast(device.type):

    print("Starting warmup")
    for _ in range(warmup):
        C = torch.bmm(X, X)
    torch.cuda.synchronize()
    print("Warmup complete")

    start_time = time()
    for _ in range(n):
        C = torch.bmm(X, X)
    torch.cuda.synchronize()
    end_time = time()
    total_time = end_time-start_time

print("Mixed Precision Multiplication")
print("----------------------")
print(f"Time per matmul: {(total_time)/n:.3f} seconds")
print(f"Flops: {n*N*2*(D**3)/total_time/1e12:2f} TFLOPs\n")

Data type: torch.float32
----------------------
Starting warmup
Warmup complete
Mixed Precision Multiplication
----------------------
Time per matmul: 0.012 seconds
Flops: 47.460368 TFLOPs



In [16]:
torch.set_default_dtype(torch.float32)

# Model Inference Runtime

In [29]:
import torchvision

In [30]:
warmup = 5
n = 20
N = 256
C = 3
H, W = 224, 224

In [31]:
model_flops = 4.09e9

### FP32

In [32]:
model = torchvision.models.resnet50().to(device)
X = torch.rand(N, C, H, W, device=device)
print(f"Data type: {X.dtype}")
print("Started warmup")
for _ in range(warmup):
    model(X)
torch.cuda.synchronize()
print("Finished warmup")

start_time = time()
for _ in range(n):
    model(X)
torch.cuda.synchronize()
end_time = time()
total_time = end_time-start_time

print("FP32 ResNet50")
print("----------------------")
print(f"Time per model evaluation: {(total_time)/n:.3f} seconds")
print(f"Flops: {n*N*model_flops/total_time/1e12:2f} TFLOPs\n")

Data type: torch.float32
Started warmup
Finished warmup
FP32 ResNet50
----------------------
Time per model evaluation: 0.209 seconds
Flops: 5.001153 TFLOPs



### FP16

In [33]:
torch.set_default_dtype(torch.float16)

In [34]:
model = torchvision.models.resnet50().to(device)
X = torch.rand(N, C, H, W, device=device)
print(f"Data type: {X.dtype}")
print("Started warmup")
for _ in range(warmup):
    model(X)
torch.cuda.synchronize()
print("Finished warmup")

start_time = time()
for _ in range(n):
    model(X)
torch.cuda.synchronize()
end_time = time()
total_time = end_time-start_time

print("FP16 ResNet50")
print("----------------------")
print(f"Time per model evaluation: {(total_time)/n:.3f} seconds")
print(f"Flops: {n*N*model_flops/total_time/1e12:2f} TFLOPs\n")

Data type: torch.float16
Started warmup
Finished warmup
FP16 ResNet50
----------------------
Time per model evaluation: 0.122 seconds
Flops: 8.559948 TFLOPs



### BF16

In [35]:
torch.set_default_dtype(torch.bfloat16)

In [36]:
model = torchvision.models.resnet50().to(device)
X = torch.rand(N, C, H, W, device=device)
print(f"Data type: {X.dtype}")
print("Started warmup")
for _ in range(warmup):
    model(X)
torch.cuda.synchronize()
print("Finished warmup")

start_time = time()
for _ in range(n):
    model(X)
torch.cuda.synchronize()
end_time = time()
total_time = end_time-start_time

print("BF16 ResNet50")
print("----------------------")
print(f"Time per model evaluation: {(total_time)/n:.3f} seconds")
print(f"Flops: {n*N*model_flops/total_time/1e12:2f} TFLOPs\n")

Data type: torch.bfloat16
Started warmup
Finished warmup
BF16 ResNet50
----------------------
Time per model evaluation: 0.125 seconds
Flops: 8.363069 TFLOPs



In [37]:
torch.set_default_dtype(torch.float32)

### TF32

In [38]:
torch.backends.cuda.matmul.allow_tf32 = True 
torch.backends.cudnn.allow_tf32 = True

In [39]:
model = torchvision.models.resnet50().to(device)
X = torch.rand(N, C, H, W, device=device)
print(f"Data type: {X.dtype}")
print("Started warmup")
for _ in range(warmup):
    model(X)
torch.cuda.synchronize()
print("Finished warmup")

start_time = time()
for _ in range(n):
    model(X)
torch.cuda.synchronize()
end_time = time()
total_time = end_time-start_time

print("TF32 ResNet50")
print("----------------------")
print(f"Time per model evaluation: {(total_time)/n:.3f} seconds")
print(f"Flops: {n*N*model_flops/total_time/1e12:2f} TFLOPs\n")

Data type: torch.float32
Started warmup
Finished warmup
TF32 ResNet50
----------------------
Time per model evaluation: 0.209 seconds
Flops: 5.000224 TFLOPs



### AMP

In [40]:
model = torchvision.models.resnet50().to(device)
X = torch.rand(N, C, H, W, device=device)
print(f"Data type: {X.dtype}")
print("Started warmup")
with torch.autocast(device.type):
    for _ in range(warmup):
        model(X)
    torch.cuda.synchronize()
    print("Finished warmup")

    start_time = time()
    for _ in range(n):
        model(X)
    torch.cuda.synchronize()
    end_time = time()
    total_time = end_time-start_time

print("Mixed Precision ResNet50")
print("----------------------")
print(f"Time per model evaluation: {(total_time)/n:.3f} seconds")
print(f"Flops: {n*N*model_flops/total_time/1e12:2f} TFLOPs\n")

Data type: torch.float32
Started warmup
Finished warmup
Mixed Precision ResNet50
----------------------
Time per model evaluation: 0.116 seconds
Flops: 9.032091 TFLOPs

